# Lab 08: Dynamic Programming

**Author:** Bo Tang

Use recursion and memoization to solve a resource-allocation problem.

## Learning Objectives

- Define stages, states, actions, and transitions.
- Compare recursive Fibonacci implementations.
- Formulate system reliability using integer programming and dynamic programming.

## Setup

See the [README](README.md#getting-started) for installation. Skip the `%pip` cell if `gurobipy` and `numpy` are already installed in this kernel.

In [ ]:
# Optional: run only if these packages are missing from this kernel.
%pip install gurobipy numpy

In [1]:
import gurobipy as gp
import numpy as np
from gurobipy import GRB

## Dynamic Programming Review

Dynamic programming (DP) solves a problem through smaller subproblems whose states summarize the information needed for future decisions. It is useful when the problem has an appropriate recursive structure and repeated subproblems.

In DP, the problem is divided into several stages, each with:
- **Stages** $n = 1, 2, \cdots, N$: A sequence of decision points.
- **State Variables** $s_n$: Describe the current state, containing all necessary information to make the next decision.
- **Decision Variables** $x_n$: The choices available at each stage, which affect the next state.
- **Transition Relations** $s_{n+1} = g_n(s_n, x_n)$: Define how the system moves from one state to another based on the decision made.
- **Immediate Return** $h_n(s_n, x_n)$: The reward or cost associated with each decision at each stage.

The solution to a dynamic programming problem typically involves **backward recursion**, where we start from the last stage and move backward to calculate the optimal solution. With these basic concepts, we can express the optimal return from stage $n$ to the end of the horizon as:
$$
f_n(s_n) = \max_{x_n} \big[ h_n(s_n, x_n) + f_{n+1}\big(g_n(s_n, x_n)\big) \big]
$$

This process allows us to calculate the optimal solution by working from the end of the problem back to the start.

1. **Initialize the Final Stage**: Start from the last stage, calculating the optimal outcome for each possible state at that stage. This involves setting a base case that defines the final return without further decisions.
   - $f_N(s_N) = \max_{x_N} \big[ h_N(s_N, x_N) \big] \quad \forall s_N$

2. **Recursive Calculation for Previous Stages**: Move backward through each stage, using the values computed for future stages recursively to determine the optimal decision at each state. By storing these values, we avoid recalculating results for the same states, increasing efficiency.
   - $f_{N-1}(s_{N-1}) = \max_{x_{N-1}} \big[  h_{N-1}(s_{N-1}, x_{N-1}) + f_{N}\big(g_{N-1}(s_{N-1}, x_{N-1})\big) \big] \quad \forall s_{N-1}$

3. **Construct the Optimal Solution**: By the time we reach the first stage, we have determined the sequence of optimal decisions and the total objective value, representing the optimal solution for the entire problem.
   - $f_1(s_1) = \max_{x_1} \big[ h_1(s_1, x_1) + f_2\big(g_1(s_1, x_1)\big) \big]$

## Recursion Review

Recursion is a problem-solving technique where a function calls itself to solve smaller instances of the same problem. It’s particularly useful when a problem can naturally be divided into similar subproblems. A recursive solution typically has:

1. **Base Case**: This stops the recursion once a condition is met.
2. **Recursive Case**: The function calls itself to solve a smaller subproblem, moving closer to the base case with each call.

In Dynamic Programming, recursion helps to break down a complex problem into simpler subproblems. However, DP often adds **memoization** to recursion, storing intermediate results to avoid redundant calculations.

### Example Problem: Fibonacci Sequence (LeetCode #509)

The Fibonacci sequence is a classic example of recursion. Each term is the sum of the two preceding terms, with the sequence starting as $0, 1, 1, 2, 3, 5, \ldots$

The recursive formula is:
$$
F(n) = F(n-1) + F(n-2)
$$

Here’s a simple Python solution that uses recursion:

In [2]:
def fibonacci(n):
    # Base cases
    if n == 0:
        return 0
    elif n == 1:
        return 1
    # Recursive case
    return fibonacci(n-1) + fibonacci(n-2)

In [3]:
import time
start = time.time()
print(fibonacci(40))
end = time.time()
print(f"{end - start:.7f} sec.")

102334155
7.5871799 sec.


While this solution is straightforward, it’s inefficient for large inputs because the function recursively calls itself twice for each non-base case:

$$
F(n) = F(n−1) + F(n−2)
$$

This results in a binary recursion tree, where each call spawns two new recursive calls. The total number of calls grows exponentially (with an upper bound of $O(2^n)$).

For example, computing $F(5)$ results in the following call tree:

```
                      F(5)
                    /      \
              F(4)           F(3)
             /    \          /   \
         F(3)    F(2)      F(2)  F(1)
        /  \     /  \      /  \
     F(2) F(1) F(1) F(0) F(1) F(0)
    /  \
F(1) F(0)
```
Each node represents a recursive function call. The number of function calls grows exponentially.

We can improve the efficiency by storing computed values, known as **memoization**, to avoid redundant calculations.

In [4]:
def fibonacci(n, memo={}):
    # Base cases
    if n == 0:
        return 0
    elif n == 1:
        return 1
    # Check if value is already computed
    if n in memo:
        return memo[n]
    # Recursive calculation with memoization
    memo[n] = fibonacci(n-1, memo) + fibonacci(n-2, memo)
    return memo[n]

In [5]:
import time
start = time.time()
print(fibonacci(40))
end = time.time()
print(f"{end - start:.7f} sec.")

102334155
0.0000091 sec.


**Result:** Both implementations return $F(40)=102334155$. Memoization avoids recomputing the same Fibonacci values. Timings depend on the machine; redefine the memoized function before timing it again, since its default dictionary retains cached values.

## Problem Statement

An electronic system consists of four components: A, B, C, and D. The system requires all four components to function, and its reliability can be increased by adding parallel units for one or more of these components. The reliability and cost for different numbers of parallel components are shown in the tables below.

### Probability of Functioning

| # of Units | Reliability of A | Reliability of B | Reliability of C | Reliability of D |
|------------|------------------|------------------|------------------|------------------|
| 1          | 0.5              | 0.6              | 0.7              | 0.5              |
| 2          | 0.6              | 0.7              | 0.8              | 0.7              |
| 3          | 0.8              | 0.8              | 0.9              | 0.9              |

### Cost

| # of Units | Cost of A | Cost of B | Cost of C | Cost of D |
|------------|-----------|-----------|-----------|-----------|
| 1          | \$100     | \$200     | \$100     | \$200     |
| 2          | \$200     | \$400     | \$300     | \$300     |
| 3          | \$300     | \$500     | \$400     | \$400     |

### Goal

Assuming the four component subsystems function independently, and given a budget of \$1,000, determine how the money should be allocated to maximize the system's reliability.

In [6]:
# Parameters
reliabilities = {
    "A": [0, 0.5, 0.6, 0.8],
    "B": [0, 0.6, 0.7, 0.8],
    "C": [0, 0.7, 0.8, 0.9],
    "D": [0, 0.5, 0.7, 0.9]
}
costs = {
    "A": [0, 100, 200, 300],
    "B": [0, 200, 400, 500],
    "C": [0, 100, 300, 400],
    "D": [0, 200, 300, 400]
}

# Indices
components = ["A", "B", "C", "D"]
units = [1, 2, 3]

### Exercise: Integer Programming Formulation

Use binary variables to select exactly one unit count per component. Enforce the \$1,000 budget and maximize the sum of log reliabilities.

Report the chosen unit counts, total cost, and product of the four selected reliabilities. The printed reliability must be the product, not the log-objective value.

In [ ]:
# Exercise: complete the preceding TODOs before running this cell.
# Create a model
m = gp.Model("System Reliability")
# Variables
x = m.addVars(components, units, vtype=GRB.BINARY, name="x")

# TODO: Set the log-reliability objective, add selection and budget constraints, and solve.

In [ ]:
# Exercise: complete the preceding TODOs before running this cell.
print("Solutions:")
obj_val = 1
for c, u in x:
    if x[c,u].x >= 1 - 1e-3:
        print("Component {}: {} units.".format(c, u))
        obj_val *= reliabilities[c][u]
print("Maximum Reliability: {:.4f}".format(obj_val))

### Exercise: Dynamic Programming Formulation

Let $f(i,b)$ be the largest reliability achievable for components $i,\ldots,N-1$ with remaining budget $b$, measured **before** choosing units for component $i$.

- **Stage:** component index $i$.
- **State:** remaining budget $b$.
- **Action:** unit count $u\in\{1,2,3\}$ for the current component.
- **Transition:** $(i,b)\rightarrow(i+1,b-c_i(u))$.
- **Contribution:** reliability $r_i(u)$, multiplied by the reliability of the remaining components.

$$
f(i,b)=\max_{u:c_i(u)\leq b}\left[r_i(u)f(i+1,b-c_i(u))\right].
$$

Use a base value of 1 when no components remain and 0 for an infeasible remaining allocation. The initial call is $f(0,1000)$. Store both the best reliability and the selected unit counts.

**Discussion questions:**

- Why is the terminal value 1 rather than 0 for a product of reliabilities?
- Which information must be included in a memoization key?

Use the table below to record the candidate choices. For each state, include the remaining budget and continuation reliability; one row per component alone is not a complete DP state table.

| Stage / Decision | 1 Unit | 2 Units | 3 Units |
|-------------------|------------------|--------------------|--------------------|
| **A**             |                  |                    |                    |
| **B**             |                  |                    |                    |
| **C**             |                  |                    |                    |
| **D**             |                  |                    |                    |

Complete the recursive function and report the allocation, cost, and reliability. Compare with the integer model. Also test a budget of 600, which permits only one unit per component, and a budget below 600, for which no complete allocation is feasible. Use a fresh memo dictionary for each independent test.

In [ ]:
# Recursive dynamic programming function
def dp(comp_ind, budget, memo={}):
    # Base case: no components left
    if comp_ind >= len(components):
        max_reliability, best_units = 1, []
        return max_reliability, best_units
    # Check if solution already computed in memo
    if (comp_ind, budget) in memo:
        return memo[comp_ind, budget]
    # Init setting
    max_reliability = 0
    best_units = []
    c = components[comp_ind]
    # Recursive case: try all unit options for the current component
    for u in units:
        cost = costs[c][u]
        # Check if feasible
        if budget >= cost:
            # TODO: recursive dp
            reliability, units_used = dp()
            # TODO: calculate current reliability
            current_reliability =
            # Get best one
            if current_reliability > max_reliability:
                max_reliability = current_reliability
                best_units = [u] + units_used
    # Memoize
    # TODO: update memory
    memo[comp_ind, budget] =
    # Return the result
    return max_reliability, best_units

In [ ]:
# Exercise: complete the preceding TODOs before running this cell.
# Solve
max_reliability, unit_configuration = dp(comp_ind=0, budget=1000)
# Solution
print("Solutions:")
obj_val = 1
for i, c in enumerate(components):
    print("Component {}: {} units.".format(c, unit_configuration[i]))
print("Maximum Reliability: {:.4f}".format(max_reliability))

## Summary

- Define the state before writing the recursion.
- Use a base value that matches the additive or multiplicative objective.
- Compare the DP allocation and objective with the integer-programming model.